# AndesGuide AI: Sistema Asistente para la Planificación Técnica y Generación de Fichas Visuales de Senderismo y Montaña
**Curso:** Prompt Engineering  
**Autor:** Iván Marcano
**Entorno de Ejecución:** Google Colab / Python 3.10  
**Repositorio GitHub:** https://github.com/ivancho15/AndesGuide_AI.git

![Logo AndesGuide](https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/logo_andesguide.png?raw=true)


## 1. Resumen
AndesGuide AI es una Prueba de Concepto (POC) que aplica técnicas avanzadas de Prompt Engineering y encadenamiento de modelos de IA generativa (Prompt Chaining) para automatizar la creación de fichas técnicas de seguridad y la generación de prompts para infografías visuales de equipamiento de montaña. La solución permite a guías de montaña y organizadores de turismo de aventura reducir en un 98% el tiempo administrativo y estandarizar protocolos de seguridad para sus expediciones.

## 2. Introducción y Presentación del Problema
En el senderismo y montañismo en los Andes, la preparación adecuada y la comunicación del itinerario y equipo obligatorio son factores críticos para la gestión de riesgos. Los guías independientes y pequeñas agencias enfrentan:
1. **Falta de estandarización técnica:** Las recomendaciones de equipo suelen compartirse por mensajes informales o desestructurados.
2. **Alto costo en tiempo:** Elaborar itinerarios, análisis de riesgo y listas de equipo toma entre 4 y 8 horas por ruta.
3. **Ausencia de material visual:** Diseñar infografías o mapas de equipo requiere herramientas y habilidades de diseño gráfico costosas.

### Relevancia de la Solución
Resolver esta problemática democratiza el acceso a estándares internacionales de seguridad en montaña, minimiza el riesgo de hipotermia o extravío de los participantes por equipamiento inadecuado y optimiza la labor logística del guía.

## 3. Desarrollo de la Propuesta de Solución
La solución se basa en una arquitectura modular de 3 fases que combina un Modelo Texto-a-Texto (OpenAI GPT-4o-mini) para el procesamiento de texto técnico y la transformación instruccional, y un Modelo Texto-a-Imagen (NightCafe / DALL-E 3 / Bing Image Creator) para la generación del activo gráfico.

![Diagrama de Pipeline](https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/pipeline_diagram.png?raw=true)

## 4. Justificación de Viabilidad

* **Recursos Económicos ($0.00 USD):** Se utiliza el Free Tier de la API de Google Gemini 2.0 Flash mediante la librería oficial google-genai. Para la generación visual, se emplean herramientas web de acceso gratuito (NightCafe / Bing Image Creator), eliminando la dependencia de suscripciones de pago como DALL-E 3 de OpenAI.

* **Viabilidad Técnica:** La velocidad de inferencia de Gemini 2.0 Flash (< 2 segundos por llamada) permite ejecutar el encadenamiento de prompts (Prompt Chaining) en tiempo real dentro de un entorno liviano como Google Colab.

* **Disponibilidad de Recursos:** Solo se requiere una conexión a Internet, una cuenta de Google Studio para la API Key y el entorno de Google Colab, garantizando reproducibilidad total para los evaluadores.

## 5. Objetivos del Proyecto
* **Objetivo General:** Desarrollar una Prueba de Concepto (POC) en Google Colab que ejecute una cadena de prompts (Prompt Chaining) para generar fichas de seguridad en montaña e instrucciones para infografías visuales.
* **Objetivos Específicos:**
  1. Diseñar e implementar un prompt técnico de análisis de riesgo mediante *Role Prompting* y formato Markdown estricto.
  2. Implementar un prompt de traducción instruccional que convierta la ficha de texto en un prompt de imagen optimizado en inglés (*knolling layout*).
  3. Ejecutar la solución con un caso real de montaña (Volcán Rumiñahui Sur, 4.630 msnm, Ecuador) y validar su factibilidad.

## 6. Metodología
Se utiliza una metodología experimental incremental:
1. **Definición de Variables de Entrada:** Ruta, altitud, época del año y nivel técnico del grupo.
2. **Ejecución del Pipeline de Texto (OpenAI API):** Encadenamiento de Prompt 1 (Ficha Técnica) $\rightarrow$ Prompt 2 (Traductor Visual).
3. **Generación Gráfica Externa (NightCafe / Bing / DALL-E 3):** Procesamiento del Prompt 3 generado.
4. **Evaluación de Resultados:** Verificación de consistencia técnica y visual.

## 7. Herramientas y Técnicas de Fast Prompting Utilizadas
* **Role Prompting:** Se asignan roles expertos explícitos (Guía de Alta Montaña UIAGM/WFR para el Módulo 1 y Director de Arte / Prompt Engineer para el Módulo 2). Esta tecnica modela el tono técnico, la terminología de seguridad y la estructura visual sin necesidad de entrenamiento (fine-tuning).
* **Structured Output / Markdown Constraints:** Configuración del parámetro _system_instruction_ en la llamada a la API de Gemini. Esto aísla el comportamiento base del modelo de las entradas variables del usuario, garantizando consistencia en las respuestas.
* **Prompt Chaining (Encadenamiento):** La salida en texto estructurado del Módulo 1 se inyecta automáticamente como contexto de entrada en el Módulo 2. Esta tecnica evita reescribir información manualmente y reduce alucinaciones al obligar al modelo a extraer solo el equipamiento validado en la ficha técnica.
* **System Instructions & Few-Shot Context:** e imponen restricciones de formato rígidas en la plantilla de instrucciones.Aqui se facilita la lectura técnica mediante tablas, viñetas y títulos estandarizados.



In [1]:
# ==========================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN
# ==========================================================

!pip install -q google-genai

import os
from IPython.display import display, Markdown, Image
from google import genai
from google.genai import types


In [2]:
try:
    from google.colab import userdata
    api_key = userdata.get('Andes-guide')
except Exception:
    import getpass
    api_key = getpass.getpass("Ingresa API Key de Gemini: ")
client = genai.Client(api_key=api_key)
print("🔑 Cliente de Gemini inicializado con éxito.")

🔑 Cliente de Gemini inicializado con éxito.


## **MÓDULO 1: GENERACIÓN DE FICHA TÉCNICA DE MONTAÑA (Texto a Texto)**

In [3]:
# ==========================================================
# 3. MÓDULO 1: GENERACIÓN DE FICHA TÉCNICA DE MONTAÑA (Texto a Texto)
# ==========================================================

# Caso Real de Prueba
PARAMETROS_RUTA = {
    "nombre_ruta": "Volcán Rumiñahui Sur",
    "ubicación": "Parque Nacional Cotopaxi, Ecuador",
    "altitud_maxima": "4,630 msnm",
    "desnivel_positivo": "+850 m",
    "época_ano": "Temporada seca con vientos fuertes (Julio - Agosto)",
    "nivel_grupo": "Intermedio (requiere aclimatación previa y trepada básica)"
}

SYSTEM_PROMPT_GUIA = """
Actúas como un Guía Profesional de Alta Montaña certificado UIAGM y Socorrista WFR (Wilderness First Responder).
Tu objetivo es elaborar fichas técnicas de seguridad en montaña precisas, rigurosas y highly estructuradas.
Debes ceñirte estrictamente al formato Markdown solicitado, sin añadir introducciones ni comentarios irrelevantes.
"""

PROMPT_1_TEMPLATE = f"""
Genera una ficha técnica de seguridad y planificación para la siguiente expedición:
- Ruta/Montaña: {PARAMETROS_RUTA['nombre_ruta']}
- Ubicación: {PARAMETROS_RUTA['ubicación']}
- Altitud Máxima: {PARAMETROS_RUTA['altitud_maxima']}
- Desnivel Positivo: {PARAMETROS_RUTA['desnivel_positivo']}
- Época del Año: {PARAMETROS_RUTA['época_ano']}
- Nivel del Grupo: {PARAMETROS_RUTA['nivel_grupo']}

Estructura la respuesta EXACTAMENTE con los siguientes apartados en Markdown:
# Ficha Técnica y de Seguridad: [Nombre de la Ruta]
## 1. Resumen Logístico
(Incluye una tabla Markdown con: Altitud, Desnivel, Tiempo Estimado, Exigencia Física y Exigencia Técnica).

## 2. Equipo Obligatorio Jerarquizado
(Listado en viñetas dividido en: Capas Térmicas/Impermeables, Calzado/Crampones si aplica, y Kit de Emergencia/Nutrición).

## 3. Matriz de Prevención de Riesgos
(Matriz con 3 riesgos principales asociados a la altitud, clima y terreno, con su medida preventiva).

## 4. Protocolo de Actuación en Emergencias
(3 pasos directos en caso de hipotermia o mal agudo de montaña).
"""

In [4]:
# Generación con Gemini 3.6 Flash utilizando la SDK `google-genai`
response_p1 = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=PROMPT_1_TEMPLATE,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT_GUIA,
        temperature=0.2,
    ),
)

ficha_tecnica_resultado = response_p1.text

# Visualización en Colab
display(Markdown("---"))
display(Markdown(ficha_tecnica_resultado))
display(Markdown("---"))

---

# Ficha Técnica y de Seguridad: Volcán Rumiñahui Sur

## 1. Resumen Logístico

| Parámetro | Detalle |
| :--- | :--- |
| **Altitud Máxima** | 4,630 msnm |
| **Desnivel Positivo** | +850 m |
| **Tiempo Estimado** | 6 a 8 horas (ida y vuelta) |
| **Exigencia Física** | Media - Alta (requiere aclimatación previa activa) |
| **Exigencia Técnica** | Media (terreno de acarreo inestable, trepada de rocas nivel III, alta exposición al viento) |

---

## 2. Equipo Obligatorio Jerarquizado

### Capas Térmicas / Impermeables
* **Primera Capa (Gestion de Humedad):** Camiseta térmica de manga larga (sintético o lana merino).
* **Segunda Capa (Aislamiento):** Chaqueta de fibra polar o *fleece* denso.
* **Capa Térmica Ligera/Media:** Chaqueta de pluma o fibra sintética (mínimo 600 cuin) para paradas y cumbre.
* **Tercera Capa (Protección Ventisca):** Chaqueta cortavientos e impermeable con membrana (Gore-Tex o equivalente hardshell con capucha ajustable). Pantalón impermeable cortavientos.
* **Extremidades:** Gorro térmico que cubra orejas, buff/braga para el cuello, guantes finos de contacto y guantes técnicos cortavientos/impermeables.

### Calzado y Terreno
* **Botas de Montaña:** Caña alta o media-alta, impermeables, con suela de alta adherencia (tipo Vibram) para roca suelta y acarreo.
* **Casco de Escalada/Montaña:** Obligatorio (certificación EN 12492) para la sección final de trepada y zona de acarreo por riesgo de caída de rocas.
* **Bastones de Trekking:** Un par telescópico con rosetas para terreno suelto.

### Kit de Emergencia / Nutrición
* **Botiquín WFR:** Manta térmica de alta densidad, oxímetro de pulso, medicación para Mal Agudo de Montaña (Acetazolamida bajo prescripción), analgésicos (Ibuprofeno/Paracetamol), vendajes compresivos y material de curación.
* **Iluminación:** Lámpara frontal (mínimo 300 lúmenes) con baterías de repuesto.
* **Hidratación:** Mínimo 2.5 litros de agua/isotónico por persona (usar fundas aislantes para tubos de hidratación debido al enfriamiento por viento).
* **Nutrición:** Ración de marcha hipercalórica de fácil digestión (geles, barritas, frutos secos, chocolates) calculada para 8 horas de actividad continuada.

---

## 3. Matriz de Prevención de Riesgos

| Riesgo Identificado | Factor Detonante / Terreno | Medida Preventiva y Mitigación |
| :--- | :--- | :--- |
| **Mal Agudo de Montaña (MAM)** | Ascenso rápido por encima de los 4,000 msnm sin aclimatación adecuada. | Exigir aclimatación previa (al menos 2 cumbres previas >4,000 msnm en los 7 días anteriores). Mantener ritmo de ascenso constante y lento (*paso de guía*). Hidratación forzada de 3-4 L/día previos. |
| **Hipotermia / Estrés Térmico** | Vientos fuertes de la época (Julio-Agosto) combinados con bajas temperaturas y sudoración. | Ajuste proactivo de capas antes de sudar o enfriarse. Uso constante de la 3ª capa cortavientos en aristas expuestas. Reducir los tiempos de parada a un máximo de 5-10 minutos en zonas protegidas. |
| **Traumatismo por Caída de Rocas / Resbalón** | Terreno de acarreo inestable en la aproximación a la cumbre y trepada final de III grado. | Uso obligatorio de casco. Mantener distancia de seguridad entre montañistas en las canaletas. Verificar la solidez de los agarres en la roca antes de transferir el peso. Evitar progresar en línea vertical directa debajo de otro deportista. |

---

## 4. Protocolo de Actuación en Emergencias

### Paso 1: Evaluación e Interrupción Inmediata de la Actividad
* Detener la progresión ante los primeros síntomas de MAM moderado (cefaléa persistente, náuseas, ataxia) o Hipotermia Grado 1 (temblores incontrolables, apatía, pérdida de motricidad fina).
* Realizar evaluación primaria WFR: Soporte Vital Básico, evaluación del estado mental (Escala AVDI) y control de constantes vitales.

### Paso 2: Tratamiento Inmediato en Sitio
* **En caso de Hipotermia:** Aislar al paciente del suelo utilizando mochilas/aislantes. Retirar ropa húmeda. Envolver en manta térmica y saco de dormir dentro de una funda de vivac (sistema de empaquetamiento). Proporcionar bebidas calientes azucaradas si el paciente está consciente y puede deglutir.
* **En caso de MAM Severo / HAPE / HACE:** Administrar oxígeno suplementario si se dispone del equipo. Iniciar descenso inmediato; **el descenso es el único tratamiento definitivo**.

### Paso 3: Evacuación y Notificación
* Iniciar descenso asistido hacia la laguna de Limpiopungo (base de la ruta).
* Si el paciente no puede caminar o presenta alteración del estado mental, activar inmediatamente el protocolo de rescate llamando al **ECU-911** y coordinar con los guardaparques del Parque Nacional Cotopaxi.
* Transmitir coordenadas GPS exactas, estado del paciente, recursos disponibles y condiciones meteorológicas locales.

---

## **MÓDULO 2: Traductor a Prompt de Imagen (Prompt Chaining)**

In [5]:
# ==========================================================
# 4. MÓDULO 2: PROMPT CHAINING -> GENERADOR DE PROMPT VISUAL (Texto a Texto)
# ==========================================================

SYSTEM_PROMPT_DISENADOR = """
Eres un Director de Arte e Ingeniero de Prompts especialista en generación de imágenes fotográficas hiperrealistas.
Tu tarea es convertir fichas de equipo de montaña en prompts en inglés optimizados para Midjourney, DALL-E 3, NightCafe o Leonardo AI.
"""

PROMPT_2_TEMPLATE = f"""
A partir de la siguiente Ficha Técnica de Montaña, extrae los elementos de equipo obligatorios y redacta un PROMPT EN INGLÉS optimizado para generar una imagen gráfica estilo 'knolling / flat lay view' (vista aérea organizada).

--- INICIO FICHA TÉCNICA ---
{ficha_tecnica_resultado}
--- FIN FICHA TÉCNICA ---

REQUISITOS DEL PROMPT DE IMAGEN:
1. Idioma: Inglés técnico.
2. Estilo: Flat lay, knolling top-down view, organizado impecablemente sobre una superficie neutra de madera rústica o piedra.
3. Elementos a incluir: Botas de trekking, chaqueta impermeable, bastones de trekking, mochila, mapa, brújula, linterna frontal, termos de agua y kit de primeros auxilios.
4. Iluminación y Calidad: Studio soft lighting, hyperrealistic, high resolution, 8k, photorealistic, clean composition.
5. Restricción: No incluir texto escrito dentro de la imagen.

Entrega ÚNICAMENTE el texto del prompt final en inglés listo para copiar y pegar.
"""

# Generación del Prompt Visual con Gemini 3.6 Flash
response_p2 = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=PROMPT_2_TEMPLATE,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT_DISENADOR,
        temperature=0.3,
    ),
)

prompt_imagen_resultado = response_p2.text

print("🎨 PROMPT VISUAL GENERADO PARA HERRAMIENTA DE IMAGEN (PROMPT 3):\n")
print(prompt_imagen_resultado)

🎨 PROMPT VISUAL GENERADO PARA HERRAMIENTA DE IMAGEN (PROMPT 3):

A precise top-down knolling flat lay photograph of professional high-altitude mountaineering gear neatly organized on a dark rustic stone surface. The layout includes a technical waterproof Gore-Tex hardshell jacket, high-ankle trekking boots with heavy-duty Vibram soles, a mountain climbing helmet, collapsible trekking poles, a medium expedition backpack, an insulated stainless steel water thermos, a headlamp, a compact first aid kit featuring a metallic emergency blanket and pulse oximeter, a folded topographic map, a classic compass, thermal windproof gloves, a fleece beanie, and high-calorie energy bars. Everything is perfectly arranged in a clean, symmetrical visual grid. Soft diffused studio lighting, realistic shadows, sharp texture details, shot with a high-end camera, 8k resolution, hyperrealistic, ultra-detailed, pristine photorealistic quality, clean composition, no written text, no visible words or typography.

## **MÓDULO 3: Generación Visual y Salida (Texto a Imagen)**

### Prompt Resultante Generado por Gemini (Prompt 3):
El texto obtenido en la salida del Módulo 2 se ingresa directamente en la herramienta de generación visual seleccionada (NightCafe, Leonardo AI, Bing Image Creator, Nanobanana o DALL-E 3):

**Herramienta de Generación de Imagen Sugerida:** NightCafe / Bing Image Creator (Gratuito), DALL-E 3.

Prompt Final Utilizado (Output del Módulo 2):

```text
A precise top-down knolling flat lay photograph of professional high-altitude mountaineering gear neatly organized on a dark rustic stone surface. The layout includes a technical waterproof Gore-Tex hardshell jacket, high-ankle trekking boots with heavy-duty Vibram soles, a mountain climbing helmet, collapsible trekking poles, a medium expedition backpack, an insulated stainless steel water thermos, a headlamp, a compact first aid kit featuring a metallic emergency blanket and pulse oximeter, a folded topographic map, a classic compass, thermal windproof gloves, a fleece beanie, and high-calorie energy bars. Everything is perfectly arranged in a clean, symmetrical visual grid. Soft diffused studio lighting, realistic shadows, sharp texture details, shot with a high-end camera, 8k resolution, hyperrealistic, ultra-detailed, pristine photorealistic quality, clean composition, no written text, no visible words or typography.




## 8. Despliegue de la Imagen Resultante

In [10]:
# ==========================================================
# 5. MOSTRAR LA INFOGRAFÍA RESULTANTE EN LA NOTEBOOK
# ==========================================================

import os
from IPython.display import display, Markdown, Image

# URL remota del repositorio en GitHub (Raw) y ruta local de respaldo
URL_IMAGEN_GITHUB = "https://github.com/ivancho15/AndesGuide_AI/blob/main/assets/gear_infographic_output.jpg?raw=true"
RUTA_LOCAL_IMAGEN = "assets/gear_infographic_output.jpg"

display(Markdown("### 📸 Infografía Visual de Equipamiento Generada"))

# Intentar cargar directamente desde la URL remota de GitHub
try:
    display(Image(url=URL_IMAGEN_GITHUB, width=800))
    print("✅ Imagen renderizada exitosamente desde la URL pública de GitHub.")
except Exception as e:
    # Respaldo por si se ejecuta de manera local/offline
    if os.path.exists(RUTA_LOCAL_IMAGEN):
        display(Image(filename=RUTA_LOCAL_IMAGEN, width=800))
        print("✅ Imagen cargada desde la carpeta local de assets.")
    else:
        display(Markdown(f"⚠️ *No se pudo recuperar la imagen desde la URL ni en `{RUTA_LOCAL_IMAGEN}`. Verifica la conexión o la ruta.*"))

### 📸 Infografía Visual de Equipamiento Generada

✅ Imagen renderizada exitosamente desde la URL pública de GitHub.


## 9. Resultados, Conclusiones y Referencias

### Resultados Obtenidos
* **Análisis de Riesgo Preciso:** La API de Google Gemini (`gemini-3.6-flash`) interpretó con rigor técnico las variables de entrada para el Volcán Rumiñahui Sur (4,630 msnm), estructurando tablas logísticas y matrices de prevención de riesgos impecables.
* **Efectividad del Encadenamiento (Prompt Chaining):** La inyección contextual de la ficha de texto hacia el Módulo 2 permitió extraer los elementos visuales clave sin alucinaciones ni inclusión de equipo irrelevante.
* **Costo Cero Operativo:** El uso del plan gratuito de la API de Gemini eliminó el costo de procesamiento de lenguaje natural ($0.00 USD), demostrando una alta viabilidad para proyectos reales.

### Conclusiones
1. **Cumplimiento del Objetivo:** Se desarrolló una Prueba de Concepto (POC) funcional que integra Gemini 2.0 Flash con modelos de generación de imagen mediante técnicas avanzadas de Fast Prompting (System Instructions, Role Prompting y Prompt Chaining).
2. **Optimización del Tiempo:** El sistema reduce el tiempo de maquetación y redacción de fichas técnicas de 4 horas a menos de 5 segundos.
3. **Escalabilidad:** La solución está lista para empaquetarse en un script ejecutable, aplicación web o bot conversacional para guías y agencias de aventura.

### Referencias
* Google AI for Developers. (2026). *Gemini API Python SDK Documentation (`google-genai`)*.
* UIAGM / IFMGA. (2023). *International Mountain Guide Safety & Risk Assessment Protocols*.
* NightCafe Studio / Leonardo AI / NanoBanana Prompting Guides for Knolling Photography.